# Cabouy - consolidation des chroniques

Trois sondes physiques, pas quatre :

| Sonde | Niveau | Conductivité | Température | Autres | Acquisition |
|---|---|---|---|---|---|
| **CTD** (Diver autonome) | oui | oui | oui | - | relevée par campagnes |
| **TROLL** (Aqua TROLL) | **non** | oui | oui | turbidité, O2, chlorophylle | exports VuSitu + rapatriée par la centrale |
| **OTT** (sonde CTD de la centrale) | oui | oui | oui | - | enregistrement continu |

La centrale rapatrie les voies du TROLL (`C2`, `T2`, `Turbi`, `O2`, `Chlorophyl`) :
ce sont les **mêmes mesures** que les exports VuSitu, un second chemin d'acquisition,
**pas une quatrième sonde**. Elles sont réunies en cellule 8, sans aucun recalage.
Ses voies `level`, `C1`, `T1` viennent de SA PROPRE sonde CTD, distincte du Diver.

Donc : **2 niveaux** (CTD, OTT) et **3 conductivités / 3 températures** (CTD, TROLL, OTT).

Deux réglages commandent tout le post-traitement :

- **cellule 10** : l'ordre de préférence des sondes, global et par période ;
- **cellule 11** : le sens des corrections de niveau, `amont` ou `aval`.

Déplacer l'**échelle** ne déplace pas le capteur : la mesure reste continue, il n'y a
pas de marche à corriger dans la série, seules les **lectures** changent de référence.
Déplacer le **capteur** crée une vraie marche : elle se corrige vers l'**aval**.

## 1. Imports

In [ ]:
import os
import re
import unicodedata
from io import StringIO

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go

## 2. Chemins d'accès

Seuls les chemins et l'identité de la station. Chaque réglage de traitement est
déclaré en tête de la cellule qui l'utilise.

In [ ]:
BASE        = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Cabouy\Gaetan"
CTD_PATH    = os.path.join(BASE, r"Données brutes\CTD")
VUSITU_PATH = os.path.join(BASE, r"Données brutes\TROLL")
OTT_PATH    = os.path.join(BASE, r"Données brutes\OTT")
BARO_PATH   = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\1 - Données BARO\Gourdon baro\Patm Calès et Thémines.xlsx"
PLUIE_PATH  = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Saint Sauveur\Gaetan\Données brutes\Pluie_BV_Ouysse.csv"

OLDDATA_PATH   = os.path.join(BASE, "Cabouy_consolide_OLD.xlsx")
UTC_CTD_PATH   = os.path.join(BASE, "UTC_CTD.xlsx")
UTC_TROLL_PATH = os.path.join(BASE, "UTC_Troll.xlsx")

PUNCTUAL_NIVEAU  = os.path.join(BASE, "punctual_measurements.xlsx")
PUNCTUAL_CONDUCT = os.path.join(BASE, "punctual_measurements_conducti.xlsx")

SORTIE_CONSOLIDE = os.path.join(BASE, "Cabouy_consolide.xlsx")
SORTIE_FINALE    = os.path.join(BASE, "Cabouy_final.xlsx")
SORTIE_SVG       = os.path.join(BASE, "Graphes.svg")

PREFIXE_CTD = "Cabouy"                      # préfixe des exports Diver
BARO_COL    = "Patm Ouysse Calès [hPa]"
PAS         = "1h"

## 3. Fonctions de lecture

Les pièges de format des trois exportateurs sont traités ici, une fois pour toutes :
en-tête Diver à une ligne variable et pied `END OF DATA`, guillemets et numéros de
série de VuSitu, sentinelle `-99999` de la centrale, virgules décimales, encodages.

La conversion UTC exige une correspondance **exacte** entre le nom du fichier et la
ligne de la table de métadonnées. Une campagne dont le nom n'y figure pas est
**ignorée**, avec un avertissement qui la nomme : c'est la table qui se corrige, pas
le code qui devine.

In [ ]:
def _sans_accents(t):
    d = unicodedata.normalize("NFKD", str(t))
    return "".join(c for c in d if not unicodedata.combining(c)).lower()


def _lire_lignes(chemin, encodages=("utf-8-sig", "utf-8", "cp1252", "latin1")):
    """Essaie plusieurs encodages au lieu d'en figer un."""
    for enc in encodages:
        try:
            with open(chemin, "r", encoding=enc) as f:
                return f.read().splitlines(), enc
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Aucun encodage ne convient pour {chemin}")


def _en_datetime(serie, dayfirst=True):
    """Essaie les formats connus, garde celui qui convertit le plus de lignes."""
    txt = serie.astype("string").str.strip()
    meilleur, n_ok = None, -1
    for fmt in ("%Y/%m/%d %H:%M:%S", "%Y-%m-%d %H:%M:%S",
                "%d/%m/%Y %H:%M:%S", "%d/%m/%Y %H:%M"):
        e = pd.to_datetime(txt, format=fmt, errors="coerce")
        if e.notna().sum() > n_ok:
            meilleur, n_ok = e, e.notna().sum()
    if n_ok < len(txt):
        s = pd.to_datetime(txt, errors="coerce", dayfirst=dayfirst)
        if s.notna().sum() > n_ok:
            meilleur = s
    return meilleur


def fichiers_correspondant(path, motif):
    """Fichiers d'un dossier contenant `motif`, triés par numéro."""
    if not os.path.isdir(path):
        print(f"  dossier absent : {path}")
        return []
    noms = [f for f in os.listdir(path)
            if motif.lower() in f.lower() and f.lower().endswith((".csv", ".txt", ".mon"))]
    return sorted(noms, key=lambda n: (int(re.findall(r"\d+", n)[0])
                                       if re.findall(r"\d+", n) else 10 ** 9, n))


def lire_fichier_CTD(nom_fichier, path=CTD_PATH):
    """Export Diver. En-tête cherchée par contenu, pied `END OF DATA` reconnu."""
    chemin = os.path.join(path, nom_fichier)
    lignes, encodage = _lire_lignes(chemin)

    entete = next((i for i, l in enumerate(lignes[:200])
                   if _sans_accents(l).lstrip("\ufeff").startswith("date/time")), None)
    if entete is None:
        raise ValueError(f"En-tête 'Date/time' introuvable dans {nom_fichier}")

    df = pd.read_csv(chemin, sep=";", encoding=encodage, skiprows=entete,
                     header=0, dtype=str, engine="python")
    df.columns = [c.strip() for c in df.columns]
    col_date = df.columns[0]

    brut = df[col_date].astype("string")
    fin = brut.map(lambda v: pd.notna(v) and "end of data" in _sans_accents(v)).fillna(False)
    df = df.loc[~(fin | brut.isna() | (brut.str.strip() == ""))].copy()

    df["Date/time"] = _en_datetime(df[col_date]).dt.round(PAS)
    for col in df.columns:
        if col not in ("Date/time", col_date):
            df[col] = pd.to_numeric(df[col].astype("string").str.strip()
                                    .str.replace(",", ".", regex=False), errors="coerce")

    # Conductivité : le facteur vient de l'UNITÉ entre crochets, pas du libellé.
    for col in list(df.columns):
        if "cond" in _sans_accents(col):
            u = re.search(r"\[([^\]]*)\]", col)
            u = _sans_accents(u.group(1)) if u else ""
            df[col] = df[col] * (1000.0 if u.startswith("ms/cm") else 1.0)
            df = df.rename(columns={col: "Cond_(µS/cm)"})
            break

    return df.loc[df["Date/time"].notna()].sort_values("Date/time").reset_index(drop=True)


#: Libellés VuSitu, numéro de série retiré, vers les noms du projet.
#: La clé est le libellé sans accents ni signe degré (voir `_cle_troll`).
NOMS_TROLL = {
    "conductivite specifique (us/cm)":        "Cond_Troll_(µS/cm)",
    "temperature (c)":                        "température_Troll_(°C)",
    "turbidite (ntu)":                        "Turbidity_Troll_(NTU)",
    "concentration rdo (mg/l)":               "O2_Troll_(mg/l)",
    "saturation rdo (%sat)":                  "O2 (%Sat)",
    "fluorescence de chlorophylle-a (rfu)":   "FluorescenceChloro_a_Troll_(RFU)",
    "concentration de chlorophylle-a (ug/l)": "ConcentrationChloro_a_(µg/l)",
}


def _cle_troll(col):
    """Libellé VuSitu ramené à une clé stable : sans numéro de série, sans
    accent, micro et degré normalisés."""
    cle = _sans_accents(re.sub(r"\s*\(\d{4,}\)\s*$", "", str(col)).strip())
    return cle.replace("\u03bc", "u").replace("\u00b5", "u").replace("\u00b0", "")


def renommer_colonnes(df):
    """Retire le numéro de série du capteur, puis applique les noms du projet.

    Le numéro entre parenthèses en fin de libellé part par regex : un
    changement de sonde ne demande donc rien.
    """
    renommage = {col: NOMS_TROLL[_cle_troll(col)]
                 for col in df.columns if _cle_troll(col) in NOMS_TROLL}
    return df.rename(columns=renommage)


def lire_fichier_vusitu(nom_fichier, path=VUSITU_PATH):
    """Export VuSitu (Aqua TROLL) : guillemets retirés, colonnes normalisées."""
    lignes, _ = _lire_lignes(os.path.join(path, nom_fichier))
    df = pd.read_csv(StringIO("\n".join(l.replace('"', "") for l in lignes)), sep=",")
    col_date = next((c for c in df.columns if "date" in _sans_accents(c)), df.columns[0])
    df["DATE"] = _en_datetime(df[col_date]).dt.round(PAS)
    return renommer_colonnes(df.drop(columns=[col_date], errors="ignore"))


#: Colonnes de la centrale OTT vers les noms du projet.
#: `level`, `c1`, `t1` viennent de SA sonde CTD ; `c2`, `t2`, `turbi`, `o2`,
#: `chlorophyl` sont le TROLL rapatrié, donc le même capteur que les exports
#: VuSitu (réunis en cellule 8).
NOMS_OTT = {
    "level":      "Niveau_CTDOTT_(cm)",
    "c1":         "Cond_CTDOTT_(µS/cm)",     "t1": "Temp_CTDOTT_(°C)",
    "c2":         "Cond_TrollOTT_(µS/cm)",   "t2": "Temp_TrollOTT_(°C)",
    "turbi":      "Turbidity_TrollOTT_(NTU)",
    "o2":         "O2_TrollOTT_(mg/l)",
    "chlorophyl": "FluorescenceChloro_a_TrollOTT_(RFU)",
}


def lire_fichier_OTT(nom_fichier, path=OTT_PATH):
    """Export de la centrale d'acquisition, déjà en UTC.

    Deux pièges traités ici : la valeur -99999 qui code l'absence de mesure,
    et les horodatages en double (médiane par horodatage).
    """
    chemin = os.path.join(path, nom_fichier)
    _, encodage = _lire_lignes(chemin)
    df = pd.read_csv(chemin, sep=";", encoding=encodage, dtype=str, engine="python")
    df.columns = [c.strip() for c in df.columns]

    col_date = df.columns[0]
    df["DATE"] = _en_datetime(df[col_date]).dt.round(PAS)
    df = df.drop(columns=[col_date])

    for col in df.columns:
        if col == "DATE":
            continue
        df[col] = pd.to_numeric(df[col].astype("string").str.strip()
                                .str.replace(",", ".", regex=False), errors="coerce")
        df.loc[df[col].isin([-99999, -9999, 9999]), col] = np.nan

    df = df.rename(columns={c: NOMS_OTT[c.lower()] for c in df.columns if c.lower() in NOMS_OTT})
    df = df.dropna(subset=["DATE"]).groupby("DATE", as_index=False).median(numeric_only=True)
    return df.sort_values("DATE").reset_index(drop=True)


def convertir_en_utc0(df, nom_fichier, metadata, col_date="Date/time",
                      col_nom="Nom fichier", col_utc="UTC Fichier"):
    """Ramène les horodatages en UTC d'après la table des campagnes.

    La correspondance sur le nom de fichier est EXACTE. Si elle échoue, la
    campagne est ignorée : une correspondance tolérante masquerait une erreur
    de saisie au lieu de la faire corriger. Le message nomme le fichier et,
    s'il en existe une, la ligne qui lui ressemble.

    Le fuseau accepte 'UTC+1', 'UTC', 1, 1.0, au lieu de ne reconnaître que
    deux chaînes exactes et de renvoyer 0 en silence pour tout le reste.
    """
    ligne = metadata.loc[metadata[col_nom] == nom_fichier, col_utc]
    if ligne.empty:
        def _nu(v):
            return re.sub(r"\.(csv|txt|mon)$", "", str(v).strip(), flags=re.IGNORECASE).lower()
        proches = [str(v) for v in metadata[col_nom].dropna() if _nu(v) == _nu(nom_fichier)]
        indice = f" (ligne la plus proche : '{proches[0]}')" if proches else ""
        raise ValueError(f"'{nom_fichier}' absent de la colonne '{col_nom}'{indice}. "
                         f"À corriger dans le fichier de métadonnées.")

    v = ligne.values[0]
    if isinstance(v, (int, float, np.integer, np.floating)) and pd.notna(v):
        decalage = float(v)
    else:
        m = re.search(r"([+-]?\d+(?:[.,]\d+)?)", str(v))
        decalage = float(m.group(1).replace(",", ".")) if m else 0.0

    df = df.copy()
    df[col_date] = df[col_date] - pd.Timedelta(hours=decalage)
    return df, decalage


#: Gamme physique de chaque type de capteur. La clé est un mot-clé cherché
#: dans le nom de colonne : une nouvelle voie est couverte sans rien ajouter.
#:
#: Le NIVEAU n'y figure pas, volontairement. Il n'a pas de gamme absolue tant
#: qu'il n'est pas calé : la compensation barométrique le laisse sur une
#: origine arbitraire. Le seuil de sonde émergée est en cellule 11, sur la
#: série calée.
GAMMES = {"cond": (30, 5000), "temp": (-2, 30),
          "turbid": (0, 4000), "o2": (0, 25), "chloro": (0, 500)}


def appliquer_gammes(df, gammes=GAMMES):
    """Écarte de chaque voie ce qui est physiquement impossible.

    À appliquer avant toute mise en priorité : une voie muette qui renvoie 0
    passerait sinon le contrôle une fois recalée sur le capteur prioritaire.
    """
    for col in df.columns:
        if not pd.api.types.is_numeric_dtype(df[col]):
            continue
        for cle, (mini, maxi) in gammes.items():
            if cle in _sans_accents(col):
                hors = (df[col] < mini) | (df[col] > maxi)
                if hors.any():
                    print(f"  {col:40s} {int(hors.sum()):6d} hors gamme [{mini}, {maxi}]")
                    df.loc[hors, col] = np.nan
                break
    return df

### Fonctions de correction

Une seule règle : **on met à NaN, on ne supprime jamais de ligne**. Sur une grille
horaire régulière, une lacune doit rester visible.

`appliquer_recalage` porte le choix demandé : `amont`, `aval` ou `tout`.

In [ ]:
def graphe(traces, titre="", ylab="", points=None, col_point=None, sortie_html=None):
    """Utilitaire Plotly. `traces` = liste de (série, nom, couleur).

    Scattergl (rendu WebGL) : une chronique horaire pluriannuelle s'affiche
    sans saturer le navigateur, contrairement à Scatter.
    """
    fig = go.Figure()
    for serie, nom, couleur in traces:
        fig.add_trace(go.Scattergl(x=serie.index, y=serie, mode="lines", name=nom,
                                   line=dict(color=couleur, width=1.3)))
    if points is not None and col_point in points.columns:
        if "Couleur" in points.columns:
            couleurs = list(points["Couleur"])
        else:
            corr = (points["Correction"] if "Correction" in points.columns
                    else pd.Series("Non", index=points.index))
            couleurs = ["red" if str(v).strip() == "Oui" else "royalblue" for v in corr]
        fig.add_trace(go.Scattergl(
            x=points["Datetime"], y=points[col_point], mode="markers",
            name="points de contrôle",
            marker=dict(color=couleurs, symbol="x", size=10)))
    fig.update_layout(title=titre, xaxis_title="Date", yaxis_title=ylab,
                      template="plotly_white", hovermode="x unified")
    if sortie_html:
        fig.write_html(sortie_html)
        print(f"Graphique sauvegardé : {sortie_html}")
    fig.show()          # pas de `return fig` : sinon Jupyter réaffiche la figure


def ecarter_periodes(serie, periodes):
    """Passe à NaN les périodes listées et retourne l'historique."""
    serie, journal = serie.copy(), []
    for debut, fin, motif in periodes:
        debut, fin = sorted([pd.to_datetime(debut), pd.to_datetime(fin)])
        m = (serie.index >= debut) & (serie.index <= fin)
        journal.append({"début": debut, "fin": fin,
                        "n écartés": int((m & serie.notna().to_numpy()).sum()),
                        "motif": motif})
        serie = serie.mask(m)
    return serie, pd.DataFrame(journal)


def appliquer_recalage(serie, date, decalage, sens):
    """Ajoute `decalage` à une PARTIE de la série, choisie par `sens`.

    'tout'  : toute la chronique. Calage global sur une référence, quand la
              mesure est continue et qu'il ne manque qu'une origine.
    'aval'  : à partir de `date` incluse. Marche RÉELLE dans la mesure :
              capteur déplacé, redescendu, réinstallé à une autre profondeur.
    'amont' : strictement avant `date`. Segment historique ramené sur la
              référence actuelle, sans toucher au présent. C'est le cas d'une
              ÉCHELLE déplacée : le capteur n'a pas bougé, la mesure est
              continue, seules les anciennes lectures changent de référence.
    """
    if sens == "tout":
        cible = np.ones(len(serie), dtype=bool)
    elif sens == "aval":
        cible = np.asarray(serie.index >= date)
    elif sens == "amont":
        cible = np.asarray(serie.index < date)
    else:
        raise ValueError(f"sens inconnu : {sens!r} (attendu : tout, aval, amont)")
    return serie.where(~cible, serie + decalage)


def valeur_au_plus_proche(serie, date, tolerance_h=1):
    """Valeur mesurée la plus proche de `date`. (nan, écart) si trop loin."""
    mesures = serie.dropna()
    if mesures.empty:
        return np.nan, np.nan
    i = mesures.index[np.abs((mesures.index - date).to_numpy()).argmin()]
    ecart_h = abs((i - date).total_seconds()) / 3600
    if ecart_h > tolerance_h:
        return np.nan, ecart_h
    return float(mesures.loc[i]), ecart_h


def recaler(serie, recalages, tolerance_h=1):
    """Applique une liste de recalages, dans l'ordre, et journalise tout.

    Chaque entrée : (date, décalage, valeur_cible, sens, motif).
    Donner SOIT `décalage` (figé, reproductible), SOIT `valeur_cible` : le
    décalage est alors calculé pour que le segment concerné passe par cette
    valeur à cette date. Pour 'amont', la valeur de référence est cherchée
    dans le segment déplacé, donc strictement avant la date.
    """
    serie, journal = serie.copy(), []
    for date, decalage, cible, sens, motif in recalages:
        date = pd.to_datetime(date) if date is not None else None
        ligne = {"date": date, "sens": sens, "valeur cible": cible,
                 "décalage": np.nan, "note": "", "motif": motif}
        if decalage is not None:
            ligne["décalage"] = round(float(decalage), 2)
            ligne["note"] = "figé"
        elif cible is None:
            ligne["note"] = "ni décalage ni valeur cible : ignoré"
            journal.append(ligne)
            continue
        elif date is None:
            ligne["note"] = "valeur cible sans date : ignoré"
            journal.append(ligne)
            continue
        else:
            segment = serie[serie.index < date] if sens == "amont" else serie
            lue, ecart_h = valeur_au_plus_proche(segment, date, tolerance_h)
            if np.isnan(lue):
                ligne["note"] = (f"aucune mesure : ignoré" if np.isnan(ecart_h)
                                 else f"mesure la plus proche à {ecart_h:.1f} h : ignoré")
                journal.append(ligne)
                continue
            decalage = float(cible) - lue
            ligne["décalage"] = round(decalage, 2)
            ligne["note"] = f"calculé ({lue:.2f} vers {float(cible):.2f})"
        serie = appliquer_recalage(serie, date, decalage, sens)
        journal.append(ligne)
    return serie, pd.DataFrame(journal)


def caler_sur_points_de_controle(serie, points, col_valeur, tolerance_h=1,
                                 sens_defaut="aval"):
    """Recale la série sur les mesures ponctuelles, en cascade.

    Colonnes lues dans `points` : 'Datetime', `col_valeur`, 'Correction'
    (Oui/Non), plus si elles existent 'Sens' (tout/aval/amont) et 'Refus'
    (motif de mise à l'écart décidé par la cellule appelante).

    CHAQUE point apparaît dans le journal, appliqué ou non, avec le motif du
    refus : un tableau vide sans explication n'est pas un résultat.
    """
    recalages, refuses = [], []
    for _, ligne in points.dropna(subset=["Datetime"]).sort_values("Datetime").iterrows():
        date, valeur = ligne["Datetime"], ligne.get(col_valeur)
        motif = str(ligne.get("Refus", "") or "").strip()
        if pd.isna(valeur):
            motif = motif or "valeur absente"
        elif str(ligne.get("Correction", "Non")).strip() != "Oui":
            motif = motif or "Correction = Non"
        if motif:
            refuses.append({"date": date, "sens": "", "valeur cible": valeur,
                            "décalage": np.nan, "note": f"non appliqué : {motif}",
                            "motif": "point de contrôle"})
            continue
        sens = str(ligne.get("Sens", sens_defaut) or sens_defaut).strip().lower()
        recalages.append((date, None, float(valeur), sens, "point de contrôle"))

    serie, journal = recaler(serie, recalages, tolerance_h)
    journal = pd.concat([journal, pd.DataFrame(refuses)], ignore_index=True)
    if not journal.empty:
        journal = journal.sort_values("date").reset_index(drop=True)
    return serie, journal

## 4. CTD : lecture, conversion UTC et compensation barométrique

`1 hPa = 1.019716 cmH2O`. Soustraire des hPa à des cmH2O injecterait 2 % des
variations barométriques dans le niveau. Mettre `HPA_EN_CMH2O = 1.0` redonne
l'ancienne formule, pour comparer.

In [ ]:
HPA_EN_CMH2O = 1.019716        # 1 hPa = 100 Pa ; 1 cmH2O = 98.0665 Pa

metadata = pd.read_excel(UTC_CTD_PATH)
baro_data = pd.read_excel(BARO_PATH)
baro_data["DATE"] = pd.to_datetime(baro_data["DATE"], errors="coerce")
baro_data = baro_data[["DATE", BARO_COL]].dropna(subset=["DATE"]).drop_duplicates("DATE")

morceaux, echecs = [], []
for nom_fichier in fichiers_correspondant(CTD_PATH, PREFIXE_CTD):
    try:
        CTD, decalage = convertir_en_utc0(lire_fichier_CTD(nom_fichier), nom_fichier, metadata)
        print(f"  {nom_fichier:45s} UTC+{decalage:g} vers UTC   ({len(CTD)} lignes)")
        m = pd.merge(CTD, baro_data, left_on="Date/time", right_on="DATE", how="left")

        # Les deux pressions dans la MÊME unité avant de soustraire.
        m["Niveau_(cm)"] = m["Pression[cmH2O]"] - m[BARO_COL] * HPA_EN_CMH2O
        m = m.rename(columns={"Température[°C]": "Temp_(°C)"})
        morceaux.append(m[["Date/time", "Niveau_(cm)", "Pression[cmH2O]",
                           BARO_COL, "Cond_(µS/cm)", "Temp_(°C)"]])
    except Exception as e:
        echecs.append((nom_fichier, str(e)))

for nom, err in echecs:
    print(f"  IGNORÉ  {nom} : {err}")

merge_ctd_df = pd.concat(morceaux, ignore_index=True).sort_values("Date/time", kind="stable")
merge_ctd_df["DATE"] = merge_ctd_df["Date/time"]
print(f"\n{len(morceaux)} campagne(s) CTD retenue(s), {len(echecs)} ignorée(s), "
      f"{len(merge_ctd_df)} enregistrements en UTC.")
if baro_data.empty:
    print("Attention : baromètre vide, le niveau n'est pas compensé.")
merge_ctd_df.head()

## 5. Raccordement à l'ancienne chronique consolidée

L'ancien fichier et les campagnes CTD sont **la même sonde**, séparées par un trou
d'exploitation. Le raccord se fait ici, une fois, avant tout le reste.

- `"jonction"` : la fin de l'ancienne série rejoint le début de la nouvelle. Continu
  par construction, c'est la méthode de la V2.
- `"recouvrement"` : médiane des écarts sur toute la période commune. Plus robuste
  au bruit, mais elle répartit la dérive au lieu de la laisser au raccord.

Mettre les décalages en dur dans `DECALAGES_RACCORD` dès qu'ils conviennent : un
décalage recalculé à chaque exécution n'est pas reproductible.

In [ ]:
METHODE_RACCORD   = "jonction"
FENETRE_RACCORD_H = 0          # demi-fenêtre moyennée ; 0 = exactement la V2

# Décalages figés, appliqués aux NOUVELLES campagnes. None = recalculé et affiché.
DECALAGES_RACCORD = {"Niveau": None, "Conductivité": None}


def _mad(x):
    return float(1.4826 * np.median(np.abs(x - np.median(x)))) if x.size > 1 else np.nan


def estimer_decalage(serie_ref, serie_a_caler, methode=METHODE_RACCORD,
                     fenetre_h=FENETRE_RACCORD_H):
    """Constante à ajouter à `serie_a_caler` pour rejoindre `serie_ref`."""
    a, n = serie_ref.dropna().sort_index(), serie_a_caler.dropna().sort_index()
    if a.empty or n.empty:
        return 0.0, np.nan, 0, "série vide"

    if methode == "recouvrement":
        commun = a.index.intersection(n.index)
        if len(commun) >= 3:
            ecarts = (a.loc[commun] - n.loc[commun]).to_numpy()
            return float(np.median(ecarts)), _mad(ecarts), int(ecarts.size), "recouvrement"
        print("  recouvrement insuffisant, bascule sur la jonction")

    fen = pd.Timedelta(hours=fenetre_h)
    avant = a[a.index >= a.index.max() - fen]
    apres = n[n.index <= n.index.min() + fen]
    if avant.empty or apres.empty:
        return 0.0, np.nan, 0, "pas de jonction exploitable"
    d = float(np.median(avant.to_numpy()) - np.median(apres.to_numpy()))
    u = float(np.hypot(_mad(avant.to_numpy()), _mad(apres.to_numpy())))
    return d, u, int(avant.size + apres.size), "jonction" + (
        " (valeur unique)" if fenetre_h == 0 else f" +/- {fenetre_h} h")


olddata_df = pd.read_excel(OLDDATA_PATH)
olddata_df["DATE"] = pd.to_datetime(olddata_df["DATE"], errors="coerce")
olddata_df = olddata_df.dropna(subset=["DATE"]).sort_values("DATE")

print(f"Dernière date ancienne : {olddata_df['DATE'].max()}")
print(f"Première date nouvelle : {merge_ctd_df['DATE'].min()}\n")

ancien_i = olddata_df.set_index("DATE")
nouveau_i = merge_ctd_df.set_index("DATE")
journal_raccord = []
for nom, col_ancien, col_nouveau, unite in [
        ("Niveau", "Niveau_(cm)", "Niveau_(cm)", "cm"),
        ("Conductivité", "Cond_CTD_(µS/cm)", "Cond_(µS/cm)", "µS/cm")]:
    d, u, n, methode = estimer_decalage(ancien_i.get(col_ancien, pd.Series(dtype=float)),
                                        nouveau_i[col_nouveau])
    if DECALAGES_RACCORD.get(nom) is not None:
        print(f"{nom:13s} : {float(DECALAGES_RACCORD[nom]):+9.2f} {unite:6s} figé "
              f"(estimation courante {d:+.2f})")
        d, methode = float(DECALAGES_RACCORD[nom]), "figé"
    else:
        print(f"{nom:13s} : {d:+9.2f} {unite:6s} (+/- {u:.2f}, n={n}, {methode})")
    journal_raccord.append({"grandeur": nom, "décalage": round(d, 2), "unité": unite,
                            "incertitude": round(u, 2) if u == u else np.nan,
                            "n": n, "méthode": methode})
    merge_ctd_df[col_nouveau] = merge_ctd_df[col_nouveau] + d
journal_raccord = pd.DataFrame(journal_raccord)

# Graphe de contrôle : 7 jours de part et d'autre de la jonction.
old_f = olddata_df[olddata_df["DATE"] >= olddata_df["DATE"].max() - pd.Timedelta(days=7)]
new_f = merge_ctd_df[merge_ctd_df["DATE"] <= merge_ctd_df["DATE"].min() + pd.Timedelta(days=7)]

fig, ax1 = plt.subplots(figsize=(11, 3))
ax1.plot(old_f["DATE"], old_f["Niveau_(cm)"], label="Niveau (ancien)", color="green")
ax1.plot(new_f["DATE"], new_f["Niveau_(cm)"], label="Niveau (nouveau, raccordé)", color="blue")
ax1.set_ylabel("Niveau (cm)")
ax2 = ax1.twinx()
ax2.plot(old_f["DATE"], old_f.get("Cond_CTD_(µS/cm)"), label="Cond. (ancien)", color="orange")
ax2.plot(new_f["DATE"], new_f["Cond_(µS/cm)"], label="Cond. (nouveau, raccordé)", color="red")
ax2.set_ylabel("Conductivité (µS/cm)")
fig.legend(loc="upper center", bbox_to_anchor=(0.5, 1.14), ncol=2)
plt.tight_layout()
plt.show()

## 6. TROLL : exports VuSitu directs

In [ ]:
metadata_troll = pd.read_excel(UTC_TROLL_PATH, sheet_name=0)

morceaux, echecs = [], []
for nom_fichier in fichiers_correspondant(VUSITU_PATH, "VuSitu"):
    try:
        df_v, decalage = convertir_en_utc0(lire_fichier_vusitu(nom_fichier), nom_fichier,
                                           metadata_troll, col_date="DATE",
                                           col_utc="UTC Fichier")
        print(f"  {nom_fichier:45s} UTC+{decalage:g} vers UTC   ({len(df_v)} lignes)")
        morceaux.append(df_v)
    except Exception as e:
        echecs.append((nom_fichier, str(e)))

for nom, err in echecs:
    print(f"  IGNORÉ  {nom} : {err}")

merge_troll_df = (pd.concat(morceaux, ignore_index=True).dropna(subset=["DATE"])
                  .sort_values("DATE", kind="stable")
                  if morceaux else pd.DataFrame(columns=["DATE"]))
print(f"\n{len(morceaux)} fichier(s) TROLL retenu(s), {len(echecs)} ignoré(s), "
      f"{len(merge_troll_df)} enregistrements en UTC.")
merge_troll_df.head()

## 7. Centrale OTT

Un seul export, déjà en UTC. Il porte trois choses qu'il ne faut pas confondre :
sa **propre sonde CTD** (`level`, `C1`, `T1`), le **TROLL rapatrié** (`C2`, `T2`,
`Turbi`, `O2`, `Chlorophyl`), et rien d'autre.

In [ ]:
morceaux = []
for nom_fichier in fichiers_correspondant(OTT_PATH, ""):
    print(f"  {nom_fichier}")
    morceaux.append(lire_fichier_OTT(nom_fichier))

merge_ott_df = (pd.concat(morceaux, ignore_index=True)
                .groupby("DATE", as_index=False).median(numeric_only=True)
                .sort_values("DATE") if morceaux else pd.DataFrame(columns=["DATE"]))

if not merge_ott_df.empty:
    print(f"\n{len(merge_ott_df)} pas de temps. Couverture par voie :")
    for c in merge_ott_df.columns:
        if c == "DATE":
            continue
        ok = merge_ott_df[c].notna()
        origine = "sonde CTD de la centrale" if "CTDOTT" in c else "TROLL rapatrié"
        if ok.any():
            print(f"  {c:38s} {int(ok.sum()):6d} pts  "
                  f"({merge_ott_df.loc[ok, 'DATE'].min():%Y-%m-%d} vers "
                  f"{merge_ott_df.loc[ok, 'DATE'].max():%Y-%m-%d})  {origine}")
merge_ott_df.head()

## 8. Assemblage : une colonne par voie, sur la grille horaire

Un bloc par **lignée de capteur**, et rien n'est fusionné ici :

1. **CTD** = ancien fichier consolidé + campagnes Diver. Même sonde, raccordée en
   cellule 5.
2. **TROLL** = colonnes TROLL de l'ancien fichier + exports VuSitu, puis les voies
   rapatriées par la centrale viennent **combler les trous sans aucun recalage** :
   c'est le même capteur, un décalage entre les deux chemins n'aurait aucun sens
   physique. L'écart médian sur le recouvrement est affiché : s'il n'est pas
   quasi nul, l'hypothèse du doublon est fausse et il faut le savoir.
3. **OTT** = la sonde CTD de la centrale, seule vraie source supplémentaire.

Sur les horodatages en double (recouvrements de campagnes), **la première valeur
gagne** : `drop_duplicates`, pas de médiane, comme la V2.

In [ ]:
# Les voies, sonde par sonde. C'est le SEUL endroit où l'instrumentation est
# décrite. Une liste de plusieurs colonnes = une seule sonde, plusieurs
# chemins d'acquisition : ils sont réunis sans recalage.
VOIES = {
    "Niveau_(cm)": {
        "OTT":   ["Niveau_CTDOTT_(cm)"],
        "CTD":   ["Niveau_CTD_(cm)"],
    },
    "Conductivité": {
        "OTT":   ["Cond_CTDOTT_(µS/cm)"],
        "TROLL": ["Cond_Troll_(µS/cm)", "Cond_TrollOTT_(µS/cm)"],
        "CTD":   ["Cond_CTD_(µS/cm)"],
    },
    "Température": {
        "OTT":   ["Temp_CTDOTT_(°C)"],
        "TROLL": ["température_Troll_(°C)", "Temp_TrollOTT_(°C)"],
        "CTD":   ["Temp _CTD(°C)"],
    },
    "Turbidité_(NTU)": {
        "TROLL": ["Turbidity_Troll_(NTU)", "Turbidity_TrollOTT_(NTU)"],
    },
    "O2_(mg/l)": {
        "TROLL": ["O2_Troll_(mg/l)", "O2_TrollOTT_(mg/l)"],
    },
    "Chlorophylle_(RFU)": {
        "TROLL": ["FluorescenceChloro_a_Troll_(RFU)", "FluorescenceChloro_a_TrollOTT_(RFU)"],
    },
}

COLONNES_CTD   = ["Niveau_CTD_(cm)", "Cond_CTD_(µS/cm)", "Temp _CTD(°C)"]
COLONNES_TROLL = ["Cond_Troll_(µS/cm)", "température_Troll_(°C)", "Turbidity_Troll_(NTU)",
                  "O2_Troll_(mg/l)", "O2 (%Sat)", "FluorescenceChloro_a_Troll_(RFU)",
                  "ConcentrationChloro_a_(µg/l)"]
COLONNES_OTT   = list(NOMS_OTT.values())


def empiler(morceaux, colonnes):
    """Empile des tableaux d'une même lignée, garde `colonnes`, première valeur
    gagne sur les horodatages en double."""
    morceaux = [m for m in morceaux if m is not None and not m.empty]
    if not morceaux:
        return pd.DataFrame(columns=["DATE"]).set_index("DATE")
    pile = pd.concat(morceaux, ignore_index=True).dropna(subset=["DATE"])
    garder = [c for c in colonnes if c in pile.columns]
    pile = pile[["DATE"] + garder].sort_values("DATE", kind="stable")
    return pile.drop_duplicates(subset="DATE", keep="first").set_index("DATE")


# 1. Lignée CTD. L'ancien fichier porte le niveau sous "Niveau_(cm)" : c'est
#    la même sonde que les campagnes, il se range donc dans Niveau_CTD_(cm).
ancien_ctd = olddata_df.rename(columns={"Niveau_(cm)": "Niveau_CTD_(cm)"})
nouveau_ctd = merge_ctd_df.rename(columns={"Niveau_(cm)": "Niveau_CTD_(cm)",
                                           "Cond_(µS/cm)": "Cond_CTD_(µS/cm)",
                                           "Temp_(°C)": "Temp _CTD(°C)"})
pile_ctd = empiler([ancien_ctd, nouveau_ctd], COLONNES_CTD)

# 2. Lignée TROLL, chemin direct (ancien fichier + exports VuSitu).
pile_troll = empiler([olddata_df, merge_troll_df], COLONNES_TROLL)

# 3. Centrale OTT.
pile_ott = empiler([merge_ott_df], COLONNES_OTT)

ignorees = sorted(set(olddata_df.columns) - set(COLONNES_CTD) - set(COLONNES_TROLL)
                  - {"DATE", "Niveau_(cm)"})
if ignorees:
    print("Colonnes de l'ancien fichier non reprises (grandeurs de synthèse, "
          "recalculées ici) :")
    for c in ignorees:
        print(f"  {c}")

# Grille horaire régulière couvrant les trois lignées.
bornes = [p.index for p in (pile_ctd, pile_troll, pile_ott) if len(p)]
debut, fin = min(i.min() for i in bornes), max(i.max() for i in bornes)
grille = pd.date_range(debut, fin, freq=PAS)
full_data = pd.DataFrame(index=grille)
full_data.index.name = "DATE"
for pile in (pile_ctd, pile_troll, pile_ott):
    for col in pile.columns:
        full_data[col] = pile[col].reindex(grille)

print(f"\n{len(full_data)} pas horaires, du {full_data.index.min():%d/%m/%Y} "
      f"au {full_data.index.max():%d/%m/%Y}")

print("\nValeurs hors gamme physique écartées :")
full_data = appliquer_gammes(full_data)


def reunir_chemins(df, colonnes, sonde, parametre, seuil_ecart=0.05):
    """Réunit les chemins d'acquisition d'une MÊME sonde, sans recalage.

    Retourne (série, nom de colonne, lignes de journal). Le premier chemin
    fait foi, les suivants comblent ses trous. L'écart médian sur le
    recouvrement est mesuré : c'est le contrôle du doublon presume.
    """
    presentes = [c for c in colonnes if c in df.columns and df[c].notna().any()]
    if not presentes:
        return None, None, []
    serie = df[presentes[0]].copy()
    lignes = [{"paramètre": parametre, "sonde": sonde, "chemin": presentes[0],
               "n pas": int(serie.notna().sum()), "n comblés": 0,
               "écart médian": np.nan, "n recouvrement": 0}]
    for c in presentes[1:]:
        commun = (serie.notna() & df[c].notna()).to_numpy()
        ecart = float((serie[commun] - df[c][commun]).median()) if commun.any() else np.nan
        trous = (serie.isna() & df[c].notna()).to_numpy()
        serie = serie.where(~trous, df[c])      # même sonde : aucun recalage
        lignes.append({"paramètre": parametre, "sonde": sonde, "chemin": c,
                       "n pas": int(df[c].notna().sum()), "n comblés": int(trous.sum()),
                       "écart médian": round(ecart, 3) if ecart == ecart else np.nan,
                       "n recouvrement": int(commun.sum())})
        reference = float(np.nanmedian(np.abs(serie.to_numpy(dtype=float))))
        if ecart == ecart and reference > 0 and abs(ecart) > seuil_ecart * reference:
            print(f"  ATTENTION  {parametre} / {sonde} : écart médian {ecart:+.2f} entre "
                  f"'{presentes[0]}' et '{c}' sur {int(commun.sum())} pas communs. "
                  f"Ces deux voies sont censées être le MÊME capteur.")
    nom = presentes[0] if len(presentes) == 1 else f"{presentes[0]} + centrale"
    if len(presentes) > 1:
        df[nom] = serie
    return serie, nom, lignes


# SONDES[paramètre][sonde] = colonne de full_data qui porte cette sonde.
print("\nRéunion des chemins d'acquisition (aucun recalage : même capteur) :")
SONDES, journal_chemins = {}, []
for parametre, sondes in VOIES.items():
    for sonde, chemins in sondes.items():
        serie, nom, lignes = reunir_chemins(full_data, chemins, sonde, parametre)
        journal_chemins += lignes
        if nom is not None:
            SONDES.setdefault(parametre, {})[sonde] = nom
journal_chemins = pd.DataFrame(journal_chemins)
display(journal_chemins)

PARAMETRES = list(VOIES)
full_data.to_excel(SORTIE_CONSOLIDE)
print(f"\nDétail capteur par capteur : {SORTIE_CONSOLIDE} ({full_data.shape[1]} colonnes)")

## 9. Comparaison des sources

À lire **avant** de fixer l'ordre de la cellule 10 : qui mesure quoi, sur quelle
période, et à quel point les sondes disent la même chose.

- le tableau de **couverture** donne, sonde par sonde, le nombre de pas et la période ;
- le tableau des **écarts** donne, deux à deux, la médiane des différences sur le
  recouvrement, sa dispersion (MAD) et le nombre de pas comparés. Une médiane
  s'annule par recalage ; une MAD forte, non : elle signale une sonde qui décroche ;
- le **graphe** superpose toutes les sondes du paramètre choisi.

In [ ]:
PARAMETRE   = "Conductivité"   # "Niveau_(cm)", "Température", "Turbidité_(NTU)", ...
PAS_GRAPHE  = None             # None = résolution horaire ; "1D" pour alléger

couverture = []
for parametre, sondes in SONDES.items():
    for sonde, col in sondes.items():
        ok = full_data[col].notna()
        couverture.append({
            "paramètre": parametre, "sonde": sonde, "colonne": col,
            "n pas": int(ok.sum()),
            "% période": round(100 * ok.sum() / len(full_data), 1),
            "début": full_data.index[ok].min() if ok.any() else pd.NaT,
            "fin": full_data.index[ok].max() if ok.any() else pd.NaT,
            "médiane": round(float(full_data.loc[ok, col].median()), 2) if ok.any() else np.nan})
couverture = pd.DataFrame(couverture)
print("Couverture par sonde :")
display(couverture)

ecarts = []
for parametre, sondes in SONDES.items():
    noms = list(sondes)
    for i, a in enumerate(noms):
        for b in noms[i + 1:]:
            sa, sb = full_data[sondes[a]], full_data[sondes[b]]
            commun = (sa.notna() & sb.notna()).to_numpy()
            d = (sa[commun] - sb[commun]).to_numpy()
            ecarts.append({"paramètre": parametre, "sonde A": a, "sonde B": b,
                           "n recouvrement": int(commun.sum()),
                           "écart médian (A-B)": round(float(np.median(d)), 2) if d.size else np.nan,
                           "MAD": round(_mad(d), 2) if d.size > 1 else np.nan})
ecarts = pd.DataFrame(ecarts)
print("Écarts entre sondes, sur leur recouvrement :")
display(ecarts)

COULEURS_SONDES = {"OTT": "#1f77b4", "TROLL": "#d62728", "CTD": "#2ca02c"}
vue = full_data.resample(PAS_GRAPHE).mean(numeric_only=True) if PAS_GRAPHE else full_data
graphe([(vue[col], f"{sonde} ({col})", COULEURS_SONDES.get(sonde, "#7f7f7f"))
        for sonde, col in SONDES[PARAMETRE].items()],
       titre=f"{PARAMETRE} : sondes disponibles"
             + (f" (moyennes {PAS_GRAPHE})" if PAS_GRAPHE else ""),
       ylab=PARAMETRE)

## 10. Ordre de fusion

Le réglage central du notebook. `ORDRE` classe les **sondes**, pas les colonnes :
la première disponible fournit la valeur, les suivantes comblent ses trous.

Toutes les sondes sont d'abord ramenées sur **une seule référence**, la première
de `ORDRE` : la sonde de secours est recalée sur la sonde retenue, jamais l'inverse.
Comme la référence est unique, changer l'ordre sur une période ne crée **aucune
marche** aux bornes de cette période.

Figer un décalage dans `DECALAGES_FIGES` dès qu'il convient : la valeur calculée est
toujours affichée à côté, pour comparaison.

In [ ]:
ORDRE = ["OTT", "TROLL", "CTD"]

# Surcharge par période : (début, fin, paramètre, ordre).
ORDRE_PERIODES = [
    # ("2021-01-01", "2021-06-01", "Conductivité", ["CTD", "TROLL", "OTT"]),
]

# (paramètre, sonde) -> décalage figé appliqué à cette sonde pour la ramener
# sur la référence. Absent = médiane des écarts sur le recouvrement.
DECALAGES_FIGES = {
    # Validé le 06/12/2024 16:00 UTC : la centrale lisait 107 cm à l'échelle.
    ("Niveau_(cm)", "CTD"): 76.86,
}


def recaler_sur_reference(df, sondes, ordre, parametre, figes):
    """Ramène chaque sonde sur la première sonde disponible de `ordre`.

    La référence n'est jamais modifiée. Retourne (dict sonde -> série recalée,
    nom de la référence, journal).
    """
    dispo = [s for s in ordre if s in sondes and df[sondes[s]].notna().any()]
    dispo += [s for s in sondes if s not in dispo and df[sondes[s]].notna().any()]
    if not dispo:
        return {}, None, []
    reference = dispo[0]
    ref = df[sondes[reference]]
    recalees = {reference: ref.copy()}
    journal = [{"paramètre": parametre, "sonde": reference, "rôle": "référence",
                "décalage": 0.0, "n recouvrement": int(ref.notna().sum()),
                "MAD": np.nan, "origine": "-"}]
    for sonde in dispo[1:]:
        s = df[sondes[sonde]]
        commun = (ref.notna() & s.notna()).to_numpy()
        d = (ref[commun] - s[commun]).to_numpy()
        calcule = float(np.median(d)) if d.size else np.nan
        if (parametre, sonde) in figes:
            decalage, origine = float(figes[(parametre, sonde)]), "figé"
            if d.size:
                origine += f" (calculé : {calcule:+.2f})"
        elif d.size:
            decalage, origine = calcule, "calculé sur le recouvrement"
        else:
            decalage, origine = 0.0, "PAS DE RECOUVREMENT : aucun recalage"
            print(f"  ATTENTION  {parametre} / {sonde} : aucun pas commun avec "
                  f"{reference}, la sonde est utilisée telle quelle.")
        recalees[sonde] = s + decalage
        journal.append({"paramètre": parametre, "sonde": sonde, "rôle": "secours",
                        "décalage": round(decalage, 2),
                        "n recouvrement": int(commun.sum()),
                        "MAD": round(_mad(d), 2) if d.size > 1 else np.nan,
                        "origine": origine})
    return recalees, reference, journal


def choisir(recalees, ordre, index):
    """Première sonde disponible de `ordre` à chaque pas de temps."""
    valeur = pd.Series(np.nan, index=index)
    origine = pd.Series(pd.NA, index=index, dtype="object")
    for sonde in list(ordre) + [s for s in recalees if s not in ordre]:
        s = recalees.get(sonde)
        if s is None:
            continue
        trous = (valeur.isna() & s.notna()).to_numpy()
        valeur = valeur.where(~trous, s)
        origine = origine.where(~trous, sonde)
    return valeur, origine


# RECALEES[paramètre][sonde] = la série de cette sonde ramenée sur la
# référence. Les cellules de correction s'en servent pour tracer les sondes
# sur le même zéro que la grandeur de synthèse.
journal_fusion, RECALEES = [], {}
for parametre, sondes in SONDES.items():
    recalees, reference, lignes = recaler_sur_reference(full_data, sondes, ORDRE,
                                                        parametre, DECALAGES_FIGES)
    journal_fusion += lignes
    RECALEES[parametre] = recalees
    valeur, origine = choisir(recalees, ORDRE, full_data.index)

    for debut, fin, param, ordre in ORDRE_PERIODES:
        if param != parametre:
            continue
        periode = np.asarray((full_data.index >= pd.to_datetime(debut))
                             & (full_data.index <= pd.to_datetime(fin)))
        v, o = choisir(recalees, ordre, full_data.index)
        valeur = valeur.where(~periode, v)
        origine = origine.where(~periode, o + " (imposé)")
        journal_fusion.append({"paramètre": parametre, "sonde": " > ".join(ordre),
                               "rôle": f"ordre imposé du {debut} au {fin}",
                               "décalage": np.nan, "n recouvrement": int(periode.sum()),
                               "MAD": np.nan, "origine": "ORDRE_PERIODES"})

    full_data[parametre] = valeur
    full_data[f"{parametre}_source"] = origine

journal_fusion = pd.DataFrame(journal_fusion)
print(f"Ordre par défaut : {' > '.join(ORDRE)}")
display(journal_fusion)
print("Pas de temps fournis par chaque sonde :")
display(pd.DataFrame({p: full_data[f"{p}_source"].value_counts() for p in PARAMETRES})
        .fillna(0).astype(int).T)

# Instantané de la fusion. Les cellules de correction en repartent
# systématiquement, au lieu de relire la colonne qu'elles viennent d'écrire :
# sans cela, les relancer cumulerait les décalages.
BRUT = full_data[PARAMETRES].copy()

## 11. Niveau : échelle limnimétrique et corrections

Deux choses différentes, à ne pas mélanger.

**L'échelle déplacée** (juin 2024) ne déplace pas le capteur. La mesure enregistrée
est continue, il n'y a **aucune marche à corriger dans la série** à cette date. Ce
sont les **lectures au carnet** qui changent de référence : celles d'avant sont sur
l'ancienne échelle, celles d'après sur la nouvelle. Elles ne sont comparables que si
l'on connaît l'écart entre les deux zéros. Tant que `DECALAGE_ECHELLES` vaut `None`,
les lectures faites sur l'autre échelle sont **tracées mais jamais utilisées** pour
corriger, et le journal le dit point par point. Le `+56.5 cm` de la V2 était un
recalage visuel sans mesure derrière : il n'est pas réintroduit ici.

**Le capteur déplacé**, lui, crée une vraie marche à sa date : elle se corrige vers
l'**aval**, dans `RECALAGES_NIVEAU`, avec `sens = "aval"`.

Le sens `"amont"` sert au cas inverse : ramener un segment **historique** sur la
référence actuelle sans toucher au présent.

In [ ]:
# ── Échelle limnimétrique ────────────────────────────────────────────────
DATE_CHANGEMENT_ECHELLE = "2024-06-01"
# Cote du zéro de la NOUVELLE échelle moins celle de l'ANCIENNE, en cm.
# None = non mesuré : les lectures de l'autre échelle sont écartées.
DECALAGE_ECHELLES = None
REFERENCE_SORTIE  = "nouvelle"        # "nouvelle" ou "ancienne"

# ── Périodes écartées : (début, fin, motif) ──────────────────────────────
PERIODES_ECARTEES_NIVEAU = [
    ("2019-10-14 17:00", "2020-02-25 17:00", "sonde déplacée"),
]

# ── Recalages de la série : (date, décalage, valeur_cible, sens, motif) ──
# sens = "tout" (calage global) | "aval" (marche réelle) | "amont" (historique)
# Donner SOIT le décalage en dur, SOIT la valeur cible à cette date.
RECALAGES_NIVEAU = [
    # ("2024-06-20 09:00", None, 112.0, "aval",  "sonde redescendue de 10 cm"),
    # (None, -18.4, None, "tout", "calage global sur la nouvelle échelle"),
]

# Seuil de sonde émergée, appliqué APRÈS calage. None = pas de seuil.
# La V2 filtrait à 33 cm, mais sur une série non calée : le seuil n'y avait
# pas de sens physique. À reposer ici si la sonde émerge réellement.
NIVEAU_MINI = None

# ── Lecture et qualification des points de contrôle ──────────────────────
points_niveau = pd.read_excel(PUNCTUAL_NIVEAU)
points_niveau["Datetime"] = pd.to_datetime(points_niveau["Jour"], dayfirst=True,
                                           errors="coerce")
date_echelle = pd.to_datetime(DATE_CHANGEMENT_ECHELLE)
if "Échelle" not in points_niveau.columns:
    points_niveau["Échelle"] = np.where(points_niveau["Datetime"] < date_echelle,
                                        "ancienne", "nouvelle")
    print(f"Pas de colonne 'Échelle' : déduite de la date du relevé "
          f"(changement le {date_echelle:%d/%m/%Y}).")

points_niveau["Refus"] = ""
points_niveau["Couleur"] = "red"
autre = points_niveau["Échelle"].astype(str).str.strip() != REFERENCE_SORTIE
if DECALAGE_ECHELLES is None:
    points_niveau.loc[autre, "Refus"] = (
        f"lecture sur l'échelle {'ancienne' if REFERENCE_SORTIE == 'nouvelle' else 'nouvelle'}, "
        f"DECALAGE_ECHELLES non mesuré")
else:
    signe = -1 if REFERENCE_SORTIE == "nouvelle" else +1
    points_niveau.loc[autre, "Hauteur (cm)"] += signe * float(DECALAGE_ECHELLES)
    print(f"{int(autre.sum())} lecture(s) converties sur l'échelle {REFERENCE_SORTIE} "
          f"({signe * float(DECALAGE_ECHELLES):+.2f} cm).")
points_niveau.loc[autre, "Couleur"] = "grey"
correction = (points_niveau["Correction"] if "Correction" in points_niveau.columns
              else pd.Series("Non", index=points_niveau.index))
points_niveau.loc[correction.astype(str).str.strip() != "Oui", "Couleur"] = "royalblue"

# ── Corrections, en repartant de BRUT ────────────────────────────────────
niveau_avant = BRUT["Niveau_(cm)"].copy()
serie, journal_periodes_niveau = ecarter_periodes(niveau_avant, PERIODES_ECARTEES_NIVEAU)
serie, journal_recalages_niveau = recaler(serie, RECALAGES_NIVEAU)
serie, journal_points_niveau = caler_sur_points_de_controle(serie, points_niveau,
                                                            "Hauteur (cm)")
if NIVEAU_MINI is not None:
    sous = (serie < NIVEAU_MINI).to_numpy()
    print(f"Seuil {NIVEAU_MINI} cm : {int(sous.sum())} pas mis à NaN (sonde émergée).")
    serie = serie.mask(sous)

full_data["Niveau_(cm)"] = serie

print("Périodes écartées :")
display(journal_periodes_niveau)
print("Recalages de la série :")
display(journal_recalages_niveau)
print("Points de contrôle (rouge = appliqué, bleu = Correction Non, gris = autre échelle) :")
display(journal_points_niveau)

# Les sondes sont tracées recalées sur la référence de la cellule 10, donc
# sur le même zéro que la série de synthèse : les écarts visibles sont réels.
graphe([(niveau_avant, "avant correction", "lightgrey"),
        (full_data["Niveau_(cm)"], "après correction", "black")]
       + [(s, f"sonde {sonde} (recalée)", COULEURS_SONDES.get(sonde, "#7f7f7f"))
          for sonde, s in RECALEES["Niveau_(cm)"].items()],
       titre="Niveau : avant et après correction", ylab="Niveau (cm)",
       points=points_niveau, col_point="Hauteur (cm)")

## 12. Conductivité : périodes écartées et points de contrôle

Même mécanique, même sens par défaut (`aval` : un point de contrôle mal daté décale
tout ce qui suit). `PERIODES_ECARTEES_AUTRES` couvre les autres paramètres.

In [ ]:
PERIODES_ECARTEES_COND = [
    # ("2021-01-01 00:00", "2021-01-31 00:00", "motif"),
]

# (début, fin, colonne, motif). La colonne peut être une voie brute ou une
# grandeur de synthèse.
PERIODES_ECARTEES_AUTRES = [
    ("2023-09-05 13:00", "2023-10-21 22:00", "Temp _CTD(°C)", "dérive sonde CTD"),
    ("2021-04-09 14:00", "2021-06-03 12:00", "O2_(mg/l)", "capteur RDO défaillant"),
]

points_cond = pd.read_excel(PUNCTUAL_CONDUCT)
points_cond["Datetime"] = pd.to_datetime(points_cond["Jour"], dayfirst=True, errors="coerce")

conduct_avant = BRUT["Conductivité"].copy()
serie, journal_periodes_cond = ecarter_periodes(conduct_avant, PERIODES_ECARTEES_COND)
full_data["Conductivité"], journal_cond = caler_sur_points_de_controle(
    serie, points_cond, "Conductivité")

print("Périodes écartées :")
display(journal_periodes_cond)
print("Points de contrôle :")
display(journal_cond)

journal_autres = []
for debut, fin, colonne, motif in PERIODES_ECARTEES_AUTRES:
    if colonne not in full_data.columns:
        print(f"  colonne inconnue, ligne ignorée : {colonne}")
        continue
    full_data[colonne], j = ecarter_periodes(full_data[colonne], [(debut, fin, motif)])
    j["colonne"] = colonne
    journal_autres.append(j)
journal_autres = (pd.concat(journal_autres, ignore_index=True)
                  if journal_autres else pd.DataFrame())
if not journal_autres.empty:
    print("Autres paramètres :")
    display(journal_autres)

graphe([(conduct_avant, "avant correction", "lightgrey"),
        (full_data["Conductivité"], "après correction", "black")]
       + [(s, f"sonde {sonde} (recalée)", COULEURS_SONDES.get(sonde, "#7f7f7f"))
          for sonde, s in RECALEES["Conductivité"].items()],
       titre="Conductivité : avant et après correction", ylab="Conductivité (µS/cm)",
       points=points_cond, col_point="Conductivité")

## 13. Filtre IQR sur la conductivité

Réglages d'origine de Cabouy : fenêtre `800h`, k = 1.5, appliqué jusqu'au
2021-02-14 seulement.

In [ ]:
FENETRE_IQR, K_IQR = "800h", 1.5
FIN_IQR = "2021-02-14 23:59"         # au-delà, le filtre ne s'applique plus


def filtre_iqr(serie, fenetre=FENETRE_IQR, k=K_IQR, min_periods=8):
    """Écarte ce qui sort de [Q1 - k*IQR, Q3 + k*IQR] sur fenêtre glissante centrée."""
    r = serie.rolling(fenetre, center=True, min_periods=min_periods)
    q1, q3 = r.quantile(0.25), r.quantile(0.75)
    iqr = q3 - q1
    return ((serie < q1 - k * iqr) | (serie > q3 + k * iqr)) & iqr.notna() & serie.notna()


ecartes_iqr = (filtre_iqr(full_data["Conductivité"]).to_numpy()
               & np.asarray(full_data.index <= pd.to_datetime(FIN_IQR)))
full_data["Conductivité_brute"] = full_data["Conductivité"]
full_data["Conductivité"] = full_data["Conductivité"].mask(ecartes_iqr)
full_data["Conductivité_Moyenne_Mobile"] = (full_data["Conductivité"]
                                            .rolling("6h", center=True).mean())

journal_iqr = pd.DataFrame([{"fenêtre": FENETRE_IQR, "k": K_IQR, "jusqu'au": FIN_IQR,
                             "n écartés": int(ecartes_iqr.sum()),
                             "% chronique": round(100 * ecartes_iqr.sum() / len(full_data), 3)}])
display(journal_iqr)

graphe([(full_data["Conductivité_brute"], "avant IQR", "lightgrey"),
        (full_data["Conductivité"], "après IQR", "royalblue"),
        (full_data["Conductivité_Moyenne_Mobile"], "moyenne mobile 6 h", "green")],
       titre="Conductivité : filtre IQR", ylab="Conductivité (µS/cm)")

## 14. Cote NGF

Deux formules, parce qu'elles ne donnent pas le même résultat.

- `"zéro"` : `cote = NIVEAU_NGF_CABOUY + Niveau_(cm) / 100`, le zéro de l'échelle
  étant à 107,6158 m NGF.
- `"V2"` : la formule de la V2, `ngf - (ngf - Niveau_(cm)) / 100`, qui équivaut à
  `106,5396 + Niveau_(cm) / 100`. Elle place donc le zéro **1,076 m plus bas**.

`"V2"` reste le défaut pour ne pas changer vos chiffres en silence. L'écart entre
les deux est affiché : à trancher avec le relevé topographique du repère.

In [ ]:
NIVEAU_NGF_CABOUY = 107.6158
FORMULE_NGF = "V2"              # "V2" ou "zéro"


def cote_ngf(niveau_cm, ngf=NIVEAU_NGF_CABOUY, formule=None):
    formule = FORMULE_NGF if formule is None else formule
    if formule == "zéro":
        return ngf + niveau_cm / 100
    return ngf - (ngf - niveau_cm) / 100


full_data["Niveau_(mNGF)"] = cote_ngf(full_data["Niveau_(cm)"])
print(f"Formule retenue : {FORMULE_NGF}")
print(f"  zéro d'échelle implicite : {cote_ngf(0):.4f} m NGF")
print(f"  écart entre les deux formules : "
      f"{cote_ngf(0, formule='zéro') - cote_ngf(0, formule='V2'):+.4f} m")
full_data[["Niveau_(cm)", "Niveau_(mNGF)"]].describe().round(3)

## 15. Interpolation des lacunes courtes

Les lacunes de moins de 12 h sont comblées, les plus longues restent des trous.
`Statut_<paramètre>` dit pour chaque pas si la valeur est **mesurée**, **interpolée**
ou **manquante**. La cote NGF se recalcule depuis le niveau interpolé, elle ne
s'interpole pas.

In [ ]:
MAX_TROU_H = 12


def interpoler_avec_statut(df, colonnes, max_trou_h=MAX_TROU_H, pas=PAS):
    """Comble les lacunes < max_trou_h et trace l'origine de chaque valeur."""
    df = df.copy()
    max_pas = int(pd.Timedelta(f"{max_trou_h}h") / pd.Timedelta(pas))
    for col in colonnes:
        origine = df[col]
        manquant = origine.isna().to_numpy()
        groupe = np.cumsum(np.r_[True, manquant[1:] != manquant[:-1]])
        tailles = pd.Series(groupe).groupby(groupe).transform("size").to_numpy()

        comble = origine.interpolate(method="time", limit_direction="both")
        comble = comble.mask(manquant & (tailles > max_pas))
        mesures = np.flatnonzero(~manquant)      # pas d'extrapolation hors plage mesurée
        if mesures.size:
            comble.iloc[:mesures[0]] = origine.iloc[:mesures[0]]
            comble.iloc[mesures[-1] + 1:] = origine.iloc[mesures[-1] + 1:]

        df[col] = comble
        df[f"Statut_{col}"] = np.where(~manquant, "Mesurée",
                                       np.where(comble.notna().to_numpy(),
                                                "Interpolée", "Manquante"))
    return df


full_data = interpoler_avec_statut(full_data, [c for c in PARAMETRES if c in full_data.columns])
full_data["Niveau_(mNGF)"] = cote_ngf(full_data["Niveau_(cm)"])
full_data["Statut_Niveau_(mNGF)"] = full_data["Statut_Niveau_(cm)"]

display(pd.DataFrame({c: full_data[f"Statut_{c}"].value_counts() for c in PARAMETRES})
        .fillna(0).astype(int).T)


def graphe_interpolation(variable):
    """Montre ce qui est mesuré et ce qui a été reconstruit."""
    statut = full_data[f"Statut_{variable}"]
    fig = go.Figure()
    fig.add_trace(go.Scattergl(x=full_data.index,
                               y=full_data[variable].where(statut == "Mesurée"),
                               mode="lines", name="mesurée",
                               line=dict(color="royalblue", width=1.2)))
    interpolee = (statut == "Interpolée").to_numpy()
    fig.add_trace(go.Scattergl(x=full_data.index[interpolee],
                               y=full_data[variable].to_numpy()[interpolee], mode="markers",
                               name=f"interpolée ({int(interpolee.sum())} pas)",
                               marker=dict(color="crimson", size=5)))
    manquante = (statut == "Manquante").to_numpy()
    if manquante.any():
        fig.add_trace(go.Scattergl(x=full_data.index[manquante],
                                   y=np.full(int(manquante.sum()), full_data[variable].min()),
                                   mode="markers",
                                   name=f"manquante ({int(manquante.sum())} pas)",
                                   marker=dict(color="lightgrey", size=3, symbol="line-ns-open")))
    fig.update_layout(title=f"{variable} : origine de chaque valeur", xaxis_title="Date",
                      yaxis_title=variable, template="plotly_white", hovermode="x unified")
    fig.show()


graphe_interpolation("Niveau_(cm)")      # ou "Conductivité", "Température"

## 16. Sauvegarde

Le fichier final ne porte **qu'un paramètre par grandeur**, avec son statut et la
sonde qui l'a fourni. Le détail capteur par capteur est dans le fichier écrit en
cellule 8. Chaque décision du notebook a son onglet de journal.

In [ ]:
PARAMETRES_FINAUX = PARAMETRES + ["Niveau_(mNGF)"]

colonnes = []
for p in PARAMETRES_FINAUX:
    colonnes += [c for c in (p, f"Statut_{p}", f"{p}_source") if c in full_data.columns]
chronique = full_data[colonnes].copy()

onglets = {
    "raccord_ancienne_chronique": journal_raccord,
    "chemins_acquisition":        journal_chemins,
    "fusion_sondes":              journal_fusion,
    "niveau_periodes_ecartees":   journal_periodes_niveau,
    "niveau_recalages":           journal_recalages_niveau,
    "niveau_points_controle":     journal_points_niveau,
    "cond_periodes_ecartees":     journal_periodes_cond,
    "cond_points_controle":       journal_cond,
    "autres_periodes_ecartees":   journal_autres,
    "filtre_iqr":                 journal_iqr,
}

with pd.ExcelWriter(SORTIE_FINALE) as writer:
    chronique.to_excel(writer, sheet_name="chronique")
    for nom, tableau in onglets.items():
        if tableau is not None and not tableau.empty:
            tableau.to_excel(writer, sheet_name=nom[:31], index=False)

print(SORTIE_FINALE)
print(f"  {len(chronique)} pas de temps x {chronique.shape[1]} colonnes")
print(f"  paramètres : {', '.join(PARAMETRES_FINAUX)}")
print(f"  journaux   : {', '.join(n for n, t in onglets.items() if t is not None and not t.empty)}")
print(f"\nDétail capteur par capteur : {SORTIE_CONSOLIDE} ({full_data.shape[1]} colonnes)")
display(chronique.head())

## 17. Graphe de synthèse

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True, gridspec_kw={"hspace": 0.05})

ax1 = axes[0]
ax1.plot(full_data.index, full_data["Niveau_(cm)"].rolling(12, center=True).mean(),
         color="lightseagreen")
ax1.set_ylabel("Niveau (cm)", color="lightseagreen")
ax1.tick_params(axis="y", labelcolor="lightseagreen")

if os.path.exists(PLUIE_PATH):
    pluie = pd.read_csv(PLUIE_PATH)
    pluie["Date"] = pd.to_datetime(pluie["Date"], errors="coerce")
    ax2 = ax1.twinx()
    ax2.bar(pluie["Date"], pluie["Precipitation (mm)"], width=0.8, color="royalblue")
    ax2.invert_yaxis()
    ax2.set_ylabel("Précipitations (mm)", color="royalblue")
    ax2.tick_params(axis="y", labelcolor="royalblue")

ax3 = axes[1]
ax3.plot(full_data.index, full_data["Conductivité_Moyenne_Mobile"], color="black")
ax3.set_ylabel("Conductivité (µS/cm)")
ax4 = ax3.twinx()
ax4.plot(full_data.index, full_data["Température"].rolling(12, center=True).mean(),
         color="crimson")
ax4.set_ylabel("Température (°C)", color="crimson")
ax4.tick_params(axis="y", labelcolor="crimson")

ax5 = axes[2]
ax5.plot(full_data.index, full_data["Turbidité_(NTU)"].rolling(24, center=True).mean(),
         color="darkorange")
ax5.set_ylabel("Turbidité (NTU)", color="darkorange")
ax5.tick_params(axis="y", labelcolor="darkorange")
ax5.set_ylim(0, 100)
ax6 = ax5.twinx()
ax6.plot(full_data.index, full_data["O2_(mg/l)"].rolling(24, center=True).mean(),
         color="darkmagenta")
ax6.set_ylabel("Oxygène (mg/L)", color="darkmagenta")
ax6.tick_params(axis="y", labelcolor="darkmagenta")
ax7 = ax5.twinx()
ax7.spines["right"].set_position(("outward", 40))
ax7.plot(full_data.index, full_data["Chlorophylle_(RFU)"].rolling(24, center=True).mean(),
         color="green")
ax7.set_ylabel("Chlorophylle (RFU)", color="green")
ax7.tick_params(axis="y", labelcolor="green")

plt.savefig(SORTIE_SVG, format="svg")
print(f"Graphique sauvegardé : {SORTIE_SVG}")
plt.show()